In [0]:
dbutils.library.restartPython()

In [0]:
import json, random, time, uuid
from datetime import datetime, timezone
from azure.eventhub import EventHubProducerClient, EventData

In [0]:
scope = "scope_mariiasirska"
conn_str = dbutils.secrets.get(scope, "eventhub-connection-string")
eh_name = dbutils.secrets.get(scope, "eventhub-name")

producer = EventHubProducerClient.from_connection_string(conn_str, eventhub_name=eh_name)

In [0]:
def make_event():
    return {
        "event_id": str(uuid.uuid4()),
        "device_id": f"dev-{random.randint(1, 50)}",
        "temperature": round(random.uniform(15, 35), 2),
        "humidity": round(random.uniform(30, 90), 2),
        "event_ts": datetime.now(timezone.utc).isoformat(),
    }

def send(duration_s=60, batch_size=100, pause_s=1.0):
    end = time.time() + duration_s
    sent = 0
    while time.time() < end:
        batch = producer.create_batch()
        for _ in range(batch_size):
            batch.add(EventData(json.dumps(make_event())))
        producer.send_batch(batch)
        sent += batch_size
        time.sleep(pause_s)
    return sent

In [0]:
print("events sent:", send(duration_s=60))

In [0]:
producer.close()